# Experiment 2: Interpolation and Double Descent

### Research Question

What happens to model generalization as the network approaches and
passes through the interpolation regime?

### Experimental Idea

Experiment 1 showed that increasing model width increased both training
and test accuracy in the tested range. However, the models did not reach
100% training accuracy.

Experiment 2 therefore investigates a wider range of model capacities
to determine where the network begins to interpolate the training data
and whether test performance exhibits double-descent behavior around
this transition.

### Experimental Configuration

- Dataset: MNIST
- Widths: 2048, 4096, 8192
- Optimizer: SGD
- Learning rate: 0.01
- Batch size: 128
- Epochs: 30
- Evaluation metrics: Training accuracy, validation accuracy, and validation loss

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import pandas as pd
import matplotlib.pyplot as plt

import time

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
transform = transforms.ToTensor()

full_train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.53MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.69MB/s]


In [ ]:
train_size = 50000
validation_size = 10000

train_dataset, validation_dataset = torch.utils.data.random_split(
    full_train_dataset,
    [train_size, validation_size],
    generator=torch.Generator().manual_seed(42)
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))

Training samples: 50000
Validation samples: 10000
Test samples: 10000


In [ ]:
batch_size = 128

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

## Model Architecture

The same fully connected architecture used in Experiment 1 is retained:

784 → w → w → w → 10

where `w` is the width of the hidden layers.

ReLU activations are used in the hidden layers.

The network width is varied to control model capacity while keeping
depth and other architectural choices fixed.

In [ ]:
class MLP(nn.Module):

    def __init__(self, width):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(28 * 28, width),
            nn.ReLU(),

            nn.Linear(width, width),
            nn.ReLU(),

            nn.Linear(width, width),
            nn.ReLU(),

            nn.Linear(width, 10)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
learning_rate = 0.01
epochs = 30

criterion = nn.CrossEntropyLoss()

## Fixed Training Configuration

- Optimizer: SGD
- Learning rate: 0.01
- Batch size: 128
- Epochs: 30
- Loss: Cross-Entropy Loss
- Activation: ReLU

Only network width is varied.

In [ ]:
def evaluate(model, loader):

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    average_loss = total_loss / len(loader)

    accuracy = 100 * correct / total

    return average_loss, accuracy

In [ ]:
def train_model(width, epochs=30):

    model = MLP(width).to(device)

    optimizer = optim.SGD(
        model.parameters(),
        lr=learning_rate
    )

    history = []

    start_time = time.time()

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)

        train_accuracy = 100 * correct / total

        validation_loss, validation_accuracy = evaluate(
            model,
            validation_loader
        )

        history.append({

            "width": width,
            "parameters": count_parameters(model),
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy

        })

        print(
            f"Width {width} | "
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Acc: {validation_accuracy:.2f}%"
        )

    elapsed = time.time() - start_time

    print(
        f"Finished width {width} "
        f"in {elapsed / 60:.2f} minutes"
    )

    return model, history

In [ ]:
widths_phase_a = [2048,4096, 8192]

phase_a_history = []

phase_a_models = {}

for width in widths_phase_a:

    print("\n" + "=" * 60)
    print(f"PHASE A: WIDTH = {width}")
    print("=" * 60)

    model, history = train_model(
        width,
        epochs=30
    )

    phase_a_history.extend(history)

    phase_a_models[width] = model


PHASE A: WIDTH = 2048
Width 2048 | Epoch 1/30 | Train Acc: 43.44% | Val Acc: 60.03%
Width 2048 | Epoch 2/30 | Train Acc: 68.93% | Val Acc: 78.19%
Width 2048 | Epoch 3/30 | Train Acc: 81.57% | Val Acc: 85.20%
Width 2048 | Epoch 4/30 | Train Acc: 86.39% | Val Acc: 87.74%
Width 2048 | Epoch 5/30 | Train Acc: 88.63% | Val Acc: 88.87%
Width 2048 | Epoch 6/30 | Train Acc: 89.75% | Val Acc: 89.72%
Width 2048 | Epoch 7/30 | Train Acc: 90.57% | Val Acc: 90.65%
Width 2048 | Epoch 8/30 | Train Acc: 91.08% | Val Acc: 91.04%
Width 2048 | Epoch 9/30 | Train Acc: 91.67% | Val Acc: 91.33%
Width 2048 | Epoch 10/30 | Train Acc: 92.08% | Val Acc: 91.82%
Width 2048 | Epoch 11/30 | Train Acc: 92.49% | Val Acc: 92.22%
Width 2048 | Epoch 12/30 | Train Acc: 92.85% | Val Acc: 92.46%
Width 2048 | Epoch 13/30 | Train Acc: 93.21% | Val Acc: 92.79%
Width 2048 | Epoch 14/30 | Train Acc: 93.52% | Val Acc: 93.03%
Width 2048 | Epoch 15/30 | Train Acc: 93.82% | Val Acc: 93.32%
Width 2048 | Epoch 16/30 | Train Acc: 94.

In [ ]:
phase_a_df = pd.DataFrame(phase_a_history)

phase_a_df.to_csv(
    "experiment_2_phase_a_results.csv",
    index=False
)

In [ ]:
phase_a_final = (
    phase_a_df
    .sort_values("epoch")
    .groupby("width")
    .tail(1)
    .sort_values("width")
)

phase_a_final

,width,parameters,epoch,train_loss,train_accuracy,validation_loss,validation_accuracy
29,2048,10020874,30,0.119164,96.668,0.156079,95.45
59,4096,36818954,30,0.107126,97.058,0.141938,95.74
89,8192,140746762,30,0.093678,97.436,0.133852,96.03


In [ ]:
phase_a_final[
    [
        "width",
        "parameters",
        "train_loss",
        "train_accuracy",
        "validation_loss",
        "validation_accuracy"
    ]
]

,width,parameters,train_loss,train_accuracy,validation_loss,validation_accuracy
29,2048,10020874,0.119164,96.668,0.156079,95.45
59,4096,36818954,0.107126,97.058,0.141938,95.74
89,8192,140746762,0.093678,97.436,0.133852,96.03


### Conclusion

Increasing the model width from 2048 to 8192 increased both training and validation accuracy, while validation loss decreased. However, the model did not reach the interpolation threshold, with training accuracy remaining below 100%.

Thus, double-descent behavior was not observed within the tested range of model capacities.